# Memory

In [1]:
# Importar Bibliotecas Necessárias
import os
from dotenv import load_dotenv
import requests
from langchain_community.llms import Ollama
from langchain_ollama import OllamaLLM  , ChatOllama 
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from IPython.display import Markdown, display

# Carregar variáveis de ambiente
load_dotenv()

print("✅ Bibliotecas importadas com sucesso!")

✅ Bibliotecas importadas com sucesso!


In [2]:
# Verificar conexão com Ollama
OLLAMA_BASE_URL = "http://localhost:11434"

try:
    response = requests.get(f"{OLLAMA_BASE_URL}/api/tags")
    if response.status_code == 200:
        modelos = response.json()
        print("✅ Conexão com Ollama estabelecida!")
        print(f"\n📦 Modelos disponíveis:")
        for modelo in modelos.get('models', []):
            print(f"  - {modelo['name']}")
    else:
        print("❌ Erro ao conectar com Ollama")
except requests.exceptions.ConnectionError:
    print("❌ Não foi possível conectar ao Ollama. Certifique-se que está rodando em Docker")

✅ Conexão com Ollama estabelecida!

📦 Modelos disponíveis:
  - olmo-3:7b
  - llama3.2:1b
  - llama3.1:8b
  - llama3.2:latest


A maioria das aplicações de Modelos de LLM possui uma interface conversacional. Um componente essencial de uma conversa é a capacidade de se referir a informações introduzidas anteriormente na conversa. No mínimo, um sistema conversacional deve ser capaz de acessar diretamente alguma janela de mensagens passadas. Um sistema mais complexo precisará ter um modelo de mundo que está constantemente atualizando, o que lhe permite fazer coisas como manter informações sobre entidades e suas relações.

Chamamos essa capacidade de armazenar informações sobre interações passadas de "Memory", ou memória. LangChain oferece muitas utilidades para adicionar memória a um sistema. Essas utilidades podem ser usadas por si só ou incorporadas de maneira integrada em uma chain.

In [1]:
from langchain_core.chat_history import InMemoryChatMessageHistory

memory = InMemoryChatMessageHistory()


In [2]:
memory.add_user_message('Olá, modelo!')
memory.add_ai_message('Olá, user')

In [3]:
memory.messages

[HumanMessage(content='Olá, modelo!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='Olá, user', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

## Criando uma conversa com memória

In [6]:
# from langchain_openai.chat_models import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "Você é um tutor de programação chamado Asimo. Responda as perguntas de forma didática."),
    ("placeholder", "{memoria}"),
    ("human", "{pergunta}"),
])
chain = prompt | ChatOllama(model="llama3.2:1b")

In [11]:
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}
def get_by_session_id(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

chain_com_memoria = RunnableWithMessageHistory(
    chain,
    get_by_session_id,
    input_messages_key='pergunta',
    history_messages_key='memoria'
)

In [12]:
config = {'configurable': {'session_id': 'usuaria_a'}}
resposta = chain_com_memoria.invoke({'pergunta': 'O meu nome é Adriano'}, config=config)

display(Markdown(resposta.content))

Olá Adriano! É um prazer conversar com você. Como posso ajudá-lo hoje? Você está procurando aprender algo novo sobre programação ou está tentando resolver um problema específico?

In [13]:
resposta = chain_com_memoria.invoke({'pergunta': 'Qual é o meu nome?'}, config=config)
resposta

AIMessage(content='Peço desculpas pelo erro anterior! O seu nome é Adriano, né? E você está gostando da aprendizagem de programação, o que? Você tem algum projeto em mente ou está procurando aprender uma habilidade nova?', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-06T20:23:14.3546827Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2762353600, 'load_duration': 94040900, 'prompt_eval_count': 112, 'prompt_eval_duration': 164691500, 'eval_count': 51, 'eval_duration': 2477404000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019cc4d1-85e8-70b2-8744-88e13836169b-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 112, 'output_tokens': 51, 'total_tokens': 163})

In [14]:
config = {'configurable': {'session_id': 'usuaria_b'}}
resposta = chain_com_memoria.invoke({'pergunta': 'Qual é o meu nome?'}, config=config)
resposta

AIMessage(content='Meu nome é João. Eu sou o tutor Asimo. Estou aqui para ajudá-lo a aprender e a crescer em programação. Qual é a sua próxima pergunta?', additional_kwargs={}, response_metadata={'model': 'llama3.2:1b', 'created_at': '2026-03-06T20:23:45.7649131Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2380285500, 'load_duration': 151455400, 'prompt_eval_count': 54, 'prompt_eval_duration': 128726800, 'eval_count': 40, 'eval_duration': 2082339000, 'logprobs': None, 'model_name': 'llama3.2:1b', 'model_provider': 'ollama'}, id='lc_run--019cc4d2-0217-7ff3-a4e4-ee461f608967-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 54, 'output_tokens': 40, 'total_tokens': 94})